# Tidal Disruption Event End-to-End Simulation

Simulates an optical/UV tidal disruption event (TDE) population (`VanVelzenTDESED`): a constant volumetric rate of 3.1e-7 Mpc⁻³ yr⁻¹ (Yao et al. 2023) out to z=2, with a 200-day duration window.

This notebook is fully self-contained. It downloads the default UVEX schedule, samples a Monte Carlo tidal disruption event population against it, builds the per-event summary table, applies a realistic set of selection cuts (repeat detections, a bright peak, and a tightly bracketed explosion time), and reports how many events UVEX should actually find, along with the distributions of their key properties. It follows the same pipeline as the `uvex-transients` CLI (see `configs/quickstart_tde.yaml` and `configs/full_run.yaml`), but driven from Python so you can poke at every intermediate object.

The notebook walks through:

1. **Sampling** a Monte Carlo population against the schedule.
2. **Screening** with the magnitude and SNR cuts.
3. The **event summary table**, built from the synthetic photometry.
4. **Selection cuts** applied as masks on that table.
5. The **detection funnel**.
6. The **expected number of events**, corrected for the Monte Carlo sampling.
7. **Distributions** of the selected events, and their sky positions.
8. An **example light curve**.

Run all cells top to bottom.


## Setup

Load the default schedule and configure the transient population.


In [ ]:
import numpy as np
from astropy import units as u
from IPython.display import Markdown, display
from matplotlib import pyplot as plt

# `uvex_transients` installs warning filters for some noisy-but-harmless dependency
# warnings (e.g. lal's Jupyter/IPython SWIG redirect notice) as soon as it's imported
# -- import it before `m4opt`/`ligo.skymap` (which trigger that warning) so the filter
# is already in place.
from uvex_transients.simulation.core import SurveySimulator
from uvex_transients.simulation.rates import estimate_yield
from uvex_transients.surveys import get_schedule
from uvex_transients.utils.plotting import add_funnel_legend, compute_funnel_bounds, plot_detection_funnel
from uvex_transients.transients.TDEs import TidalDisruptionEvent

from uvex_transients.missions import uvex_fast as uvex

schedule = get_schedule()
tde = TidalDisruptionEvent()
transients = {"tde": tde}
simulator = SurveySimulator(schedule, transients=transients, simulation_seed=42)

LABELS = {"tde": "TDE"}
COLORS = {"tde": "#55A868"}

print(tde)
print(f"redshift_limit={tde.redshift_limit}, duration_limit={tde.duration_limit}")

## Sample a Monte Carlo population

`generate_events` uses **windowed sampling**: it only draws events within HEALPix pixels/time
bins the schedule could plausibly have caught, given the transient's own duration window.


In [ ]:
# TDEs are a comparatively high-rate population out to z=2 -- a freshly-sampled catalog over a full-mission schedule can run to hundreds of thousands of (mostly far-too-faint) events, so draw a random 1/20th subset instead and rescale counts back up afterward.
TIME_BINS = 200
NSIDE = 256
# None (no downsampling), one factor for every population, or a dict mapping each transient key to its own factor.
DOWNSAMPLE = 20
# Per-population factor every count below is multiplied back up by (1 for a population that is not downsampled).
SCALE = {key: (DOWNSAMPLE.get(key) if isinstance(DOWNSAMPLE, dict) else DOWNSAMPLE) or 1 for key in transients}

catalog = simulator.generate_events(time_bins=TIME_BINS, nside=NSIDE, downsample=DOWNSAMPLE)
n_sampled = {key: int(np.sum(catalog.transient_type == key)) * SCALE[key] for key in transients}
print(f"Sampled {sum(n_sampled.values()):,} events across {TIME_BINS} time bin(s) at NSIDE={NSIDE}.")

## Screen the population

The freshly sampled catalog holds every event of the population that exploded inside the survey's footprint and time window, and almost all of them are far too faint for UVEX to see. Two cuts from `SurveySimulator` do the first screening, before anything expensive is computed:

- `filter_by_limiting_magnitude`: could the event ever clear 25 AB mag? This is cheap and independent of the schedule, so it removes hopeless events early.
- `filter_by_snr`: was the event detected above SNR 5 by at least one real observation of its position?

All further selection is done on a per-event summary table, built next.

In [ ]:
MAG_LIMIT = 25.0  # AB mag
SNR_THRESHOLD = 5.0  # a measurement above this SNR counts as a detection

mag_filtered = simulator.filter_by_limiting_magnitude(catalog, uvex, mag_limit=MAG_LIMIT)
detected = simulator.filter_by_snr(mag_filtered, uvex, snr_threshold=SNR_THRESHOLD)
print(
    f"{len(catalog):,} generated, {len(mag_filtered):,} could clear {MAG_LIMIT:g} AB mag, {len(detected):,} detected."
)

## Photometry and the event summary table

For the detected events, `run_photometry_action` simulates the full synthetic photometry (every observation, in every band), and `run_event_summary_action` reduces it, together with the schedule's epochs, to one row per event. The columns used below are:

| Column | Meaning |
|---|---|
| `n_det` | Number of epochs after the explosion with SNR above threshold. |
| `t_first_det` | Time of the first detection. |
| `t_last_constraining_nondet` | The last epoch before the first detection that is a non-detection and also *constraining*: the rise from it to the first detection is at least 3 sigma, so a marginal non-detection does not count. It can precede the explosion, and it is masked if the first observation was already a detection. |
| `peak_mag` | AB magnitude of the highest-SNR measurement in the photometry. |
| `weight` | Expected number of real UVEX events this one Monte Carlo event stands for, $\mu_0 / n$ for its type. |
| `redshift`, `coord`, ... | Copied from the event catalog. |

The table's `meta` also records the number of events generated per type ($n$), the intrinsic expected number of UVEX events ($\mu_0$, from the exposure catalog) and the rate uncertainty. That is everything `estimate_yield` needs to turn any selection of rows into an expected yield.

In [ ]:
exposure = simulator.compute_effective_exposure(time_bins=TIME_BINS, nside=NSIDE)
photometry = simulator.run_photometry_action(detected, uvex)
summary = simulator.run_event_summary_action(
    catalog=detected,
    exposure=exposure,
    photometry=photometry,
    mission=uvex,
    snr_threshold=SNR_THRESHOLD,
)

print(f"{len(summary):,} events in the summary table, with {len(summary.colnames)} columns.")
summary[
    "event_id", "transient_type", "redshift", "n_det", "t_first_det", "t_last_constraining_nondet", "peak_mag", "weight"
][:5]

## Selection cuts

The science selection is three boolean masks on the summary table, applied one after another. Because they are plain masks, any of them is easy to change or reorder.

| Cut | Requirement |
|---|---|
| Repeat detections | `n_det` is at least 2. A lone detection is hard to tell apart from noise or an artifact. |
| Bright peak | `peak_mag` is below 23 AB mag. |
| Bracketed explosion | `t_first_det - t_last_constraining_nondet` is below a per-population limit, so the explosion time is pinned down. An event with no constraining non-detection has no such bracket and is dropped. |

The allowed gap is set at roughly the time each event takes to rise: the explosion should be known to better than that. The thresholds are plain constants at the top of the next cell.

| Population | Max gap (days) | Reasoning |
|---|--:|---|
| TDE | 30 | TDEs rise over a few weeks, so a month brackets the rise itself. |

Masked entries (no constraining non-detection, no peak magnitude) are stored as 0 underneath the mask, so they are converted to NaN first. That makes them fail every comparison instead of slipping through.

In [ ]:
MIN_DETECTIONS = 2  # minimum number of detected epochs
PEAK_MAG_LIMIT = 23.0  # AB mag; the peak measurement must be brighter than this

# Longest allowed gap (days) between the last constraining non-detection and the first detection.
MAX_NONDETECTION_GAP = {"tde": 30.0}


def filled(column):
    """A masked column as a plain float array, with masked entries as NaN so that they fail every comparison."""
    return np.ma.filled(np.ma.asarray(column, dtype=float), np.nan)


types = np.asarray(summary["transient_type"]).astype(str)
n_det = np.asarray(summary["n_det"])
peak_mag = filled(summary["peak_mag"])
redshift = filled(summary["redshift"])
weight = filled(summary["weight"])
gap = np.ma.filled((summary["t_first_det"] - summary["t_last_constraining_nondet"]).to_value(u.day), np.nan)
t_first = np.ma.filled((summary["t_first_det"] - summary["t_explosion"]).to_value(u.day), np.nan)
max_gap = np.array([MAX_NONDETECTION_GAP[key] for key in types], dtype=float)

# Each cut is a boolean mask over the rows of the summary table, applied cumulatively.
with np.errstate(invalid="ignore"):
    repeated = n_det >= MIN_DETECTIONS
    bright = repeated & (peak_mag < PEAK_MAG_LIMIT)
    selected = bright & (gap <= max_gap)

stage_names = [
    "Generated",
    f"Could clear {MAG_LIMIT:g} AB mag",
    f"Detected (SNR > {SNR_THRESHOLD:g})",
    f"At least {MIN_DETECTIONS} detections",
    f"Peak brighter than {PEAK_MAG_LIMIT:g} mag",
    "Explosion bracketed (selected)",
]
stage_counts = {
    key: [
        summary.meta["n_pre_cut"][key],
        int(np.sum(mag_filtered.transient_type == key)),
        int(np.sum(types == key)),
        int(np.sum((types == key) & repeated)),
        int(np.sum((types == key) & bright)),
        int(np.sum((types == key) & selected)),
    ]
    for key in transients
}

width = max(len(label) for label in LABELS.values()) + 2
print(f"{'Monte Carlo events per stage':<36s}" + "".join(f"{LABELS[key]:>{width}s}" for key in transients))
for i, name in enumerate(stage_names):
    print(f"{name:<36s}" + "".join(f"{stage_counts[key][i]:>{width},}" for key in transients))


def fmt(x):
    """Format a count with thousands separators, or 3 significant figures when small."""
    return f"{x:,.0f}" if x >= 100 else f"{x:.3g}"

## Detection funnel

How many events of each population survive each stage, expressed as **expected UVEX events**: the number of Monte Carlo events that reach a stage times $\mu_0 / n$ for their type. The first bar is therefore the intrinsic number of events $\mu_0$, and the downsampling and windowed sampling are already accounted for.

Black error bars are the **MC (statistical)** uncertainty on each stage, treating it as a binomial subsample of the generated events (Clopper-Pearson). The pale shaded band is the **rate (systematic)** uncertainty from each population's own literature rate normalization (`RATE_CI`), which scales every stage by the same factor rather than shrinking as the sample is cut down.

In [ ]:
funnel_labels = [
    "Generated",
    f"Could clear\n{MAG_LIMIT:g} AB mag",
    f"Detected\n(SNR > {SNR_THRESHOLD:g})",
    f"At least {MIN_DETECTIONS}\ndetections",
    f"Peak brighter\nthan {PEAK_MAG_LIMIT:g} mag",
    "Explosion\nbracketed",
]

fig, ax = plt.subplots(figsize=(10, 4.5))
bar_width = 0.8 / len(transients)
positive = []
for i, key in enumerate(transients):
    raw = stage_counts[key]
    # One Monte Carlo event stands for mu_0 / n real events, so this puts every stage in expected UVEX events.
    per_event = summary.meta["expected_events"][key] / max(summary.meta["n_pre_cut"][key], 1)
    mc_lower, mc_upper, rate_lower, rate_upper = compute_funnel_bounds(raw, rate_ci=summary.meta["rate_ci"][key])
    counts = np.asarray(raw, dtype=float) * per_event
    x = np.arange(len(funnel_labels)) + (i - (len(transients) - 1) / 2) * bar_width
    plot_detection_funnel(
        ax,
        x=x,
        counts=counts,
        mc_lower=mc_lower * per_event,
        mc_upper=mc_upper * per_event,
        rate_lower=rate_lower * per_event,
        rate_upper=rate_upper * per_event,
        color=COLORS[key],
        width=bar_width,
        label=LABELS[key],
    )
    tops = np.maximum(rate_upper, mc_upper) * per_event
    positive += [v for v in np.concatenate([counts, tops]) if v > 0]
    for xi, count, top in zip(x, counts, tops):
        ax.text(xi, max(top, min(positive) / 3), fmt(count), ha="center", va="bottom", fontsize=7)
ax.set_xticks(np.arange(len(funnel_labels)), funnel_labels)
ax.set_yscale("log")
ax.set_ylim(min(positive) / 3, max(positive) * 30)  # headroom for the legend above the bars
ax.set_ylabel("Expected UVEX events")
ax.set_title("Detection funnel")
add_funnel_legend(ax)
fig.tight_layout()

## Expected number of events

The Monte Carlo counts alone are not a forecast. The sampler is windowed (it only draws events where and when the schedule could catch them) and usually downsampled, so raw counts depend on the sampling settings. `estimate_yield` converts the selection into a physical expectation, per transient type, from three numbers in the table's `meta` and the mask:

- $n$, the number of events generated.
- $k$, the number of rows the mask selects.
- $\mu_0$, the intrinsic number of events that should explode inside UVEX's footprint and observing window, from the footprint-aware exposure.

The fraction selected is $\hat\epsilon = k / n$ and the expected yield is $\hat\lambda = \mu_0 \hat\epsilon$. Downsampling is uniform, so it cancels in $k / n$, and $\mu_0$ does not depend on how many events were sampled, which corrects for both effects. There are two independent uncertainties: a Monte Carlo one (Clopper-Pearson interval on $k / n$) and a rate one (the literature `RATE_CI` on $\mu_0$). Types are reported separately, never summed, since that would mix independent Monte Carlo errors with a shared rate uncertainty. If nothing is selected, only an upper limit is meaningful.

In [ ]:
yields = estimate_yield(summary, mask=selected)


def show_expected_events(yields):
    """Render the per-population yield estimate as a Markdown table."""
    ci = f"{yields.meta['confidence']:.0%}"
    lines = [
        f"| Population | Intrinsic UVEX events | Selected / generated | Fraction ({ci} CI) | **Expected selected events** |",
        "|---|--:|--:|--:|--:|",
    ]
    for row in yields:
        key = str(row["transient_type"])
        fraction = (
            f"{row['fraction']:.2%} ({row['fraction_lower']:.2%} to {row['fraction_upper']:.2%})"
            if row["n_pre_cut"]
            else "n/a"
        )
        expected = row["expected_events"]
        if row["n_selected"] == 0:
            result = f"**< {fmt(row['expected_events_binom_upper'])}** ({ci} upper limit)"
        else:
            up_mc, lo_mc = row["expected_events_binom_upper"] - expected, expected - row["expected_events_binom_lower"]
            up_rate, lo_rate = (
                row["expected_events_rate_upper"] - expected,
                expected - row["expected_events_rate_lower"],
            )
            result = f"**{fmt(expected)}** (+{fmt(up_mc)} / -{fmt(lo_mc)} MC, +{fmt(up_rate)} / -{fmt(lo_rate)} rate)"
        lines.append(
            f"| {LABELS[key]} | {fmt(row['intrinsic_events'])} | {row['n_selected']} / {row['n_pre_cut']} "
            f"| {fraction} | {result} |"
        )
    display(Markdown("\n".join(lines)))


show_expected_events(yields)

## Distributions of the selected events

Histograms of the properties that drive each cut, plus the redshift, all read from the summary table. Each event is weighted by its `weight` column, so bin heights are **expected numbers of UVEX events** and the legend entries are totals: the gray histogram adds up to the expected number of events detected at least once, and the colored one to the expected yield. Dashed lines mark the cut thresholds.

- **Peak magnitude**: the AB magnitude of the highest-SNR measurement. Events to the left of the line survive. It is fainter than the true peak of the light curve whenever the cadence misses the maximum.
- **Number of detections**: events to the right of the line survive.
- **Days from explosion to first detection**: how long after the explosion UVEX first catches the event, in the observer frame.
- **Days from last constraining non-detection to first detection**: how tightly the survey brackets the explosion. Events to the left of the line survive, and those with no constraining non-detection are left out of this panel.

Each panel shows a single cut, so some events on the surviving side of a line are still removed by one of the others.

In [ ]:
PANELS = {
    "redshift": ("Redshift", redshift, False, None),
    "peak_mag": ("Peak magnitude (AB)", peak_mag, False, lambda key: PEAK_MAG_LIMIT),
    "n_det": (f"Number of detections (SNR > {SNR_THRESHOLD:g})", n_det.astype(float), True, lambda key: MIN_DETECTIONS),
    "t_first": ("Days from explosion to first detection", t_first, False, None),
    "gap": (
        "Days from last constraining non-detection to first detection",
        gap,
        True,
        lambda key: MAX_NONDETECTION_GAP[key],
    ),
}


def make_bins(values, log=False, integer=False, n=25):
    if values.size == 0:
        return np.linspace(0, 1, 5)
    lo, hi = values.min(), values.max()
    if integer:
        return np.unique(np.round(np.geomspace(1, hi + 1, n)))
    if log:
        lo = max(lo, 1e-3)
        return np.geomspace(lo, max(hi, 2 * lo), n)
    return np.linspace(lo, hi, n) if hi > lo else np.linspace(lo - 0.5, hi + 0.5, n)


def draw_panel(ax, quantity, key):
    xlabel, values, log_x, cut = PANELS[quantity]
    usable = (types == key) & np.isfinite(values)
    chosen = usable & selected
    ax.set_xlabel(xlabel)
    if not usable.any():
        ax.text(0.5, 0.5, "no events", transform=ax.transAxes, ha="center", va="center")
        ax.set_xticks([])
        ax.set_yticks([])
        return
    bins = make_bins(values[usable], log=log_x, integer=quantity == "n_det")
    ax.hist(
        values[usable], bins=bins, weights=weight[usable], color="0.8", label=f"Detected ({fmt(weight[usable].sum())})"
    )
    ax.hist(
        values[chosen],
        bins=bins,
        weights=weight[chosen],
        color=COLORS[key],
        label=f"Selected ({fmt(weight[chosen].sum())})",
    )
    if cut is not None:
        ax.axvline(cut(key), color="k", ls="--", lw=1, label="Cut")
    ax.set_yscale("log")
    if log_x:
        ax.set_xscale("log")
    ax.legend(fontsize=7)


if len(transients) == 1:
    (key,) = transients
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    for ax, quantity in zip(axes.flat, PANELS):
        draw_panel(ax, quantity, key)
    for ax in axes[:, 0]:
        ax.set_ylabel("Expected UVEX events per bin")
    axes.flat[-1].axis("off")
    fig.suptitle(f"{LABELS[key]}: distributions")
    fig.tight_layout()
else:
    for quantity in PANELS:
        fig, axes = plt.subplots(1, len(transients), figsize=(4.5 * len(transients), 3.6), sharey=True)
        for ax, key in zip(axes, transients):
            draw_panel(ax, quantity, key)
            ax.set_title(LABELS[key])
        axes[0].set_ylabel("Expected UVEX events per bin")
        fig.tight_layout()

## Sky distribution

Where the selected events fall on the sky, against everything that was generated.

In [ ]:
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(111, projection="aitoff")
ax.grid(True)

ax.scatter(
    catalog.coord.ra.wrap_at(180 * u.deg).radian,
    catalog.coord.dec.radian,
    s=2,
    alpha=0.2,
    color="#888888",
    label=f"Generated ({len(catalog):,})",
)
for key in transients:
    # The rows of the summary table follow the order of `detected`.
    chosen = selected & (types == key)
    coord = detected.coord[chosen]
    ax.scatter(
        coord.ra.wrap_at(180 * u.deg).radian,
        coord.dec.radian,
        s=8,
        color=COLORS[key],
        label=f"{LABELS[key]} selected ({int(chosen.sum()):,})",
    )
ax.legend(loc="lower right", markerscale=2, fontsize=7)
ax.set_title("Sky distribution (Monte Carlo events)")
fig.tight_layout()

## An example light curve

Reconstruct one event as a real `Event` (`EventCatalog.get_events`) and run full synthetic photometry (`Event.simulate_photometry`) against every observation the schedule actually made of it. Prefer an event that passed every cut, but the number that do depends on the specific random draw above, so fall back to the deepest available stage if none did. Filled squares are detections, open triangles are upper limits, and the lines are the noiseless model.

In [ ]:
rng = np.random.default_rng(1)
selected_ids = detected.event_id[selected]
for _label, _source, _ids in (
    ("selected", detected, selected_ids),
    ("detected", detected, detected.event_id),
    ("mag-screened", mag_filtered, mag_filtered.event_id),
    ("sampled", catalog, catalog.event_id),
):
    if len(_ids) > 0:
        example_catalog, example_ids, example_stage = _source, _ids, _label
        break
else:
    raise RuntimeError("No events at all were sampled -- try a larger TIME_BINS/NSIDE or a smaller DOWNSAMPLE.")

example_id = rng.choice(example_ids)
event = example_catalog.get_events(int(example_id), {"tde": tde}, schedule)
print(f"Example event (from the {example_stage!r} stage):")
print(event)

phot = event.simulate_photometry(uvex)
t_since_explosion = (phot["obs_time"] - event.t_explosion).to(u.day)
t_theory = np.linspace(0, tde.duration_limit.to_value(u.day), 300) * u.day

fig, ax = plt.subplots(figsize=(7, 4))
for band, color in {"FUV": "#4C72B0", "NUV": "#DD8452"}.items():
    ax.plot(t_theory.value, event.mag(t_theory, uvex, band=band).value, color=color, lw=1.5, alpha=0.6)

    in_band = np.isfinite(phot["ab_mag"]) & (phot["band"] == band)
    detected_pts = in_band & (phot["snr"] > SNR_THRESHOLD)
    upper_limits = in_band & (phot["snr"] <= SNR_THRESHOLD)

    if np.any(detected_pts):
        ax.errorbar(
            t_since_explosion[detected_pts].value,
            phot["ab_mag"][detected_pts],
            yerr=5 * phot["mag_err"][detected_pts],
            marker="s",
            mfc=color,
            mec="k",
            ecolor=color,
            linestyle="none",
            label=band,
        )
    if np.any(upper_limits):
        ax.errorbar(
            t_since_explosion[upper_limits].value,
            phot["ab_mag"][upper_limits],
            yerr=[
                phot["mag_upper"][upper_limits] - phot["ab_mag"][upper_limits],
                np.abs(phot["mag_lower"][upper_limits] - phot["ab_mag"][upper_limits]),
            ],
            marker="v",
            mfc="w",
            mec=color,
            ecolor=color,
            linestyle="none",
        )

ax.invert_yaxis()
ax.set_xlabel("Days since explosion")
ax.set_ylabel("AB magnitude")
ax.set_title(f"Event {event.event_id} (z={event.redshift:.3f}, {event.n_observations} observations)")
ax.legend()
fig.tight_layout()
plt.show()